In [ ]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%load_ext autoreload
%autoreload 2

import random
import numpy as np
import pandas as pd
import torch

import Python.config as cfg
import Python.utils as ut
import Python.simfun as sim
import Python.model2 as md
import Python.metric as me
import Python.bnn_mcmc as mcmc
import Python.bnn_train as train

seed = 123

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

X, y, feature_true, signal, sim_info = sim.simfun_nonlinear(
    n=160,
    p=1,
    n_active=1,
    interaction=False,
    seed=seed,
    device=device,
)

In [ ]:
# -----------------------------
# 1. Split data and true signal
# -----------------------------

split_cfg = cfg.SplitConfig(
    train_frac=0.60,
    val_frac=0.20,
    test_frac=0.20,
    seed=seed,
)

indices = ut.make_split(X.shape[0], split_cfg)
splits = ut.split_data(X, y, indices, mode="tensor")
signal_splits = ut.split_data(X, signal, indices, mode="tensor")

X_train = splits["X_train"]
y_train = splits["y_train"]
X_test = splits["X_test"]
signal_test = signal_splits["y_test"]

x_val_grid = torch.linspace(
    -torch.pi,
    torch.pi,
    201,
    device=device,
    dtype=X_train.dtype,
)[:, None]
signal_val_grid = torch.cos(x_val_grid[:, 0])

x_grid = torch.linspace(
    -torch.pi,
    torch.pi,
    400,
    device=device,
    dtype=X_train.dtype,
)[:, None]
signal_grid = torch.cos(x_grid[:, 0])

In [ ]:
# -----------------------------
# 2. Direct-unit configuration
# -----------------------------

direct_arch = dict(
    H=3,
    gate_roles=("input", "breakpoint", "output"),
    gate_power=2.0,
    gate_tau=1.0,
)

experiments = pd.DataFrame([
    {"experiment": "A",  "parameterization": "direct unit", "gate": "none", "gate_roles": (), "init_sd": 0.5},
    {"experiment": "B1", "parameterization": "direct unit", "gate": "input+breakpoint+output", "gate_roles": direct_arch["gate_roles"], "init_sd": 0.082},
    {"experiment": "B2", "parameterization": "direct unit", "gate": "input+breakpoint+output", "gate_roles": direct_arch["gate_roles"], "init_sd": 0.25},
    {"experiment": "B3", "parameterization": "direct unit", "gate": "input+breakpoint+output", "gate_roles": direct_arch["gate_roles"], "init_sd": 0.5},
    {"experiment": "B4", "parameterization": "direct unit", "gate": "input+breakpoint+output", "gate_roles": direct_arch["gate_roles"], "init_sd": 1.0},
    {"experiment": "C",  "parameterization": "residual edge-level", "gate": "all current gates", "gate_roles": None, "init_sd": 0.5},
])

experiments

In [ ]:
# -----------------------------
# 3. MCMC reference for B1--B4
# -----------------------------

mcmc_model = md.DirectUnitBNNVI(
    X=X_train,
    y=y_train,
    family=sim_info["family"],
    sigma2=sim_info["sigma2"],
    init_sd=0.5,
    K_flow=0,
    **direct_arch,
).to(device)

mcmc_ref = mcmc.run_bnn_mcmc(
    model=mcmc_model,
    N=10000,
    S_max=100,
    burnin=2000,
    thin=1,
    seed=seed,
    print_every=500,
)

mcmc_xi = torch.as_tensor(
    mcmc_ref["xi_draws"],
    device=device,
    dtype=X_train.dtype,
)

In [ ]:
# -----------------------------
# 4. Train B3
# -----------------------------

out = train.train_direct_bnn(
    X_train=X_train,
    y_train=y_train,
    X_eval=x_val_grid,
    signal_eval=signal_val_grid,
    X_final=X_test,
    signal_final=signal_test,
    mcmc_decoder=mcmc_model.decoder,
    mcmc_xi=mcmc_xi,
    family=sim_info["family"],
    sigma2=sim_info["sigma2"],
    init_sd=0.5,
    K_flow=8,
    flow_hidden_units=64,
    flow_hidden_layers=2,
    scale_clip=1.5,
    epochs=6000,
    lr=3e-4,
    R_train=100,
    R_eval=1000,
    R_final=5000,
    eval_every=250,
    seed=seed,
    **direct_arch,
)

In [ ]:
# -----------------------------
# 5. Selected checkpoint
# -----------------------------

history = out["history"]
best_id = history["val_signal_r2"].idxmax()

print(
    history.loc[
        best_id,
        [
            "epoch",
            "expected_log_likelihood",
            "kl_q_prior",
            "val_signal_r2",
            "val_draw_r2_median",
            "val_zero_function_prob",
            "val_constant_function_prob",
            "expected_path_count",
            "cancellation_ratio_median",
        ],
    ]
)

In [ ]:
from IPython.display import display

print("===== Overall metrics =====")
display(
    pd.DataFrame([out["final"]["summary"]])
    .T.rename(columns={0: "value"})
)

print("\n===== Role diagnostics =====")
display(out["final"]["role_metrics"])

print("\n===== Ordered unit paths =====")
display(out["final"]["unit_metrics"])

print("\n===== Unit contribution energy =====")
display(out["final"]["contribution_metrics"])

print("\n===== Spike and conditional slab =====")
me.print_spike_slab_summary(out["final"]["summary"])
display(out["final"]["spike_slab_metrics"])

In [ ]:
# Every checkpoint is retained in these four tables.

display(out["history"])
display(out["role_history"])
display(out["unit_history"])
display(out["contribution_history"])

In [ ]:
# -----------------------------
# 6. Function posterior plot
# -----------------------------

mcmc_grid = me.predict_draws(
    mcmc_model.decoder,
    x_grid,
    mcmc_xi,
)

rat_grid = me.predict_draws(
    out["model"].decoder,
    x_grid,
    out["final"]["xi"],
)

fig, ax = me.plot_function_1d(
    x=x_grid[:, 0].cpu(),
    signal=signal_grid.cpu(),
    mcmc_pred_draws=mcmc_grid,
    rat_pred_draws=rat_grid,
)

For experiment A, set `gate_roles=()` in both `mcmc_model` and `train_direct_bnn`. For the role-isolation runs, fix one gate open by omitting that role: input fixed uses `("breakpoint", "output")`; output fixed uses `("input", "breakpoint")`; breakpoint fixed uses `("input", "output")`. B1--B4 share the same all-gated MCMC reference because `init_sd` changes only the variational initialization.